### 1-میزان فروش روزانه ی ما چقدر است
### 2-این فروش در دسته بندی های مختلف چقدر است
### 3-فروش در یک سال آینده به چه صورت خواهد بود

### import data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

In [ ]:
data_path = os.getenv('DATA_PATH')
products = pd.read_csv(f'{data_path}/BaSalam.products.csv')
reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv')
labeled_comments = pd.read_csv(f"{data_path}/all_labeled_comments.csv")
map_category = pd.read_csv(f"{data_path}/category_map.csv")
category_mapping = dict(zip(map_category['sub_category'], map_category['main_category']))
products['main_category'] = products['categoryTitle'].map(category_mapping).fillna("سایر")
products[['categoryTitle','main_category']].sample()

### 1- میزان فروش روزانه ی ما چقدر است

- There isn't any data about daily sales so i use weekly sales mean

In [ ]:
products['sales_count_week'].sum()/7

In [ ]:
data = pd.DataFrame({'mean': products['sales_count_week'].sum()//7}, index=['-'])
fig = px.bar(data, range_y=[14000, 16000])
fig.update_layout(
    title='Daily Sales',
    xaxis_title='Sales',
    yaxis_title='Amount',
    showlegend=False,   
)
fig.show()

In [ ]:
fig = go.Figure()

fig.add_annotation(
    text=f"<b>{products['sales_count_week'].sum()//7}</b>",  
    x=0.5, y=0.5,  
    showarrow=False,
    font=dict(size=100)  
)

fig.update_layout(
    title='Daily Sales',    
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False),
    # template="simple_white",
    height=300 
)

fig.show()

### 2-این فروش در دسته بندی های مختلف چقدر است

In [ ]:
products[['main_category', 'categoryTitle']].sample(4)

In [ ]:
products[products['main_category'].str.contains('سایر')]['categoryTitle'].unique()

In [ ]:
products['daily_sales'] = products['sales_count_week'].map(lambda x: x/7)

In [ ]:
data = products.groupby("main_category")['daily_sales'].sum().reset_index().sort_values(by='daily_sales', ascending=False)
fig = px.bar(data, x='main_category', y='daily_sales')
fig.show()

In [ ]:
def map_color(high_star):
    if high_star >2000:
        return 'Above 2000' 
    elif high_star > 1000:
        return 'Between 2000-1000'
    else :
        return 'Below 1000'
data['color'] = data['daily_sales'].map(map_color)

In [ ]:
fig = px.bar(data, x='main_category', y='daily_sales', color='color', title="sales in different categories",
             color_discrete_map={'Above 2000': 'green', 'Between 2000-1000': 'lightgray', 'Below 1000': 'red'})

fig.add_hline(y=2000, line_dash="dash", line_color="black", line_width=2)
fig.add_hline(y=1000, line_dash="dash", line_color="black", line_width=2)
fig.update_xaxes(tickfont=dict(family='Arial', size=12, color='black', weight='bold'),)

fig.show()

In [ ]:
fig = px.pie(data, names='main_category', values='daily_sales')
fig.show()

### 3-فروش در یک سال آینده به چه صورت خواهد بود

- There aren't any daily sales for time series analysis
- I use comments for finding amount of satisfaction of user during time

In [ ]:
high_star = reviews[['updatedAt', 'star']]
high_star['updatedAt'] = pd.to_datetime(high_star['updatedAt']).dt.date
high_star = high_star[high_star['star'] > 3].groupby('updatedAt')['star'].size().reset_index()
high_star = high_star.iloc[:-30]
high_star.head(2)

In [ ]:
fig = px.line(high_star, x='updatedAt', y='star', title='Number of High Star Comments Over Time')
fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Number of High Star Comments'
)
fig.show()

In [ ]:
high_star['Rolling_Mean'] = high_star['star'].rolling(window=30).mean()
high_star['Rolling_Std'] = high_star['star'].rolling(window=30).std()

fig = go.Figure()

fig.add_trace(go.Scatter(x=high_star['updatedAt'], y=high_star['star'],
                        mode='lines', name='Original count', line=dict(color='lightgray')))
fig.add_trace(go.Scatter(x=high_star['updatedAt'], y=high_star['Rolling_Mean'],
                        mode='lines', name='Rolling Mean (Trend)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=high_star['updatedAt'], y=high_star['Rolling_Std'],
                        mode='lines', name='Rolling Std Dev', line=dict(color='green')))

fig.update_layout(
    title="Trend and Variance of High stars Over Time",
    xaxis_title="Date",
    yaxis_title="Value",
    legend=dict(x=0, y=1),
    height=500,
    width=900
)
fig.show()

In [ ]:
low_star = reviews[['updatedAt', 'star']]
low_star['updatedAt'] = pd.to_datetime(low_star['updatedAt']).dt.date
low_star = low_star[low_star['star'] <= 3].groupby('updatedAt')['star'].size().reset_index()
low_star = low_star.iloc[:-20]

In [ ]:
fig = px.line(low_star, x='updatedAt', y='star', title='Number of Low Star Comments Over Time')
fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Number of High Star Comments'
)
fig.show()

In [ ]:
low_star['Rolling_Mean'] = low_star['star'].rolling(window=30).mean()
low_star['Rolling_Std'] = low_star['star'].rolling(window=30).std()

fig = go.Figure()

fig.add_trace(go.Scatter(x=low_star['updatedAt'], y=low_star['star'],
                        mode='lines', name='Original count', line=dict(color='lightgray')))
fig.add_trace(go.Scatter(x=low_star['updatedAt'], y=low_star['Rolling_Mean'],
                        mode='lines', name='Rolling Mean (Trend)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=low_star['updatedAt'], y=low_star['Rolling_Std'],
                        mode='lines', name='Rolling Std Dev', line=dict(color='green')))


fig.update_layout(
    title="Trend and Variance of Low stars Over Time",
    xaxis_title="Date",
    yaxis_title="Value",
    legend=dict(x=0, y=1),
    height=500,
    width=900
)
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=low_star['updatedAt'], y=low_star['Rolling_Mean'],
                        mode='lines', name='Low stars(Trend)', line=dict(color='red')))

fig.add_trace(go.Scatter(x=high_star['updatedAt'], y=high_star['Rolling_Mean'],
                        mode='lines', name='High stars(Trend)', line=dict(color='green')))

fig.update_layout(
    title="Comparison of High vs Low stars trend Over Time",
    xaxis_title="Date",
    yaxis_title="Value",
    legend=dict(x=0, y=1),
    height=500,
    width=900
)
fig.show()

In [ ]:
high_star['updatedAt'] = pd.to_datetime(high_star['updatedAt'])
low_star['updatedAt'] = pd.to_datetime(low_star['updatedAt'])

high_star['year_month'] = high_star['updatedAt'].dt.to_period('M')
low_star['year_month'] = low_star['updatedAt'].dt.to_period('M')

high_star_monthly = high_star.groupby('year_month')['star'].sum().reset_index()
low_star_monthly = low_star.groupby('year_month')['star'].sum().reset_index()

total_stars_monthly = high_star_monthly.merge(low_star_monthly, on='year_month', suffixes=('_high', '_low'))
total_stars_monthly['total'] = total_stars_monthly['star_high'] + total_stars_monthly['star_low']

total_stars_monthly['percent_high'] = total_stars_monthly['star_high'] / total_stars_monthly['total'] * 100
total_stars_monthly['percent_low'] = total_stars_monthly['star_low'] / total_stars_monthly['total'] * 100

fig = go.Figure()

fig.add_trace(go.Scatter(x=total_stars_monthly['year_month'].astype(str), y=total_stars_monthly['percent_high'],
                        mode='lines', name='High stars(%)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=total_stars_monthly['year_month'].astype(str), y=total_stars_monthly['percent_low'],
                        mode='lines', name='Low stars(%)', line=dict(color='red')))

fig.update_layout(
    title="Percentage of High vs Low stars per Month",
    xaxis_title="Month",
    yaxis_title="Percentage",
    height=500,
    width=900
)
fig.show()

In [ ]:
merged_comments = pd.merge(labeled_comments, reviews, on='_id', suffixes=('_unlabeled', '_labeled'))
merged_comments.shape

In [ ]:
merged_comments['updatedAt'] = pd.to_datetime(merged_comments['updatedAt'])

merged_comments['year_month'] = merged_comments['updatedAt'].dt.to_period('M')

sentiment_counts = merged_comments.groupby(['year_month', 'sentiment_by_parsbert']).size().unstack(fill_value=0)

sentiment_counts['total'] = sentiment_counts.sum(axis=1)

sentiment_counts['percent_positive'] = (sentiment_counts['Positive'] / sentiment_counts['total']) * 100
sentiment_counts['percent_negative'] = (sentiment_counts['Negative'] / sentiment_counts['total']) * 100

fig = go.Figure()

fig.add_trace(go.Scatter(x=sentiment_counts.index.astype(str), y=sentiment_counts['percent_positive'],
                        mode='lines', name='Positive (%)', line=dict(color='green')))
fig.add_trace(go.Scatter(x=sentiment_counts.index.astype(str), y=sentiment_counts['percent_negative'],
                        mode='lines', name='Negative (%)', line=dict(color='red')))

fig.update_layout(
    title="Percentage of Positive, Neutral, and Negative Comments per Month",
    xaxis_title="Month",
    yaxis_title="Percentage",
    height=500,
    width=900,
)

fig.show()